# Prompt Engineering with Hugging face

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../'))
from src.rag_pipeline import RAGPipeline
from src.seed_chroma import seed_database

c:\Users\dagic\OneDrive\Documents\KAIM\Week_7\rag-complaint-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
load_dotenv()

# ----------------------------------------------------------------
# Paste your free HuggingFace token here
# Get one at: https://huggingface.co/settings/tokens
# ----------------------------------------------------------------
HF_TOKEN = os.getenv("HF_TOKEN")
# OR load from environment variable (safer — token not visible in notebook):
# HF_TOKEN = os.getenv("HF_TOKEN", "")
# ----------------------------------------------------------------

MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Connect to HuggingFace Inference API — no download, runs on HF servers
client = InferenceClient(
    model=MODEL,
    token=HF_TOKEN,
)


def ask(prompt, max_new_tokens=200):
    """
    Send a prompt to the Qwen model via HuggingFace Inference API.
    Returns the answer as a plain string.

    No model is loaded locally — the request goes to HuggingFace's servers.
    """
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
    )
    return response.choices[0].message.content.strip()


# Quick connection test
print(f"Model : {MODEL}")
print("Testing connection to HuggingFace...")
test = ask("In one sentence, what does a financial analyst do?")
print(f"Answer: {test}")
print()
print("Ready! All prompts in this notebook will use this model.")

Model : Qwen/Qwen2.5-7B-Instruct
Testing connection to HuggingFace...
Answer: A financial analyst evaluates financial data and market trends to provide investment advice and support decision-making for businesses or individuals.

Ready! All prompts in this notebook will use this model.


In [3]:
import pandas as pd

# Load the cleaned tickets CSV
df = pd.read_parquet("../data/raw/complaint_embeddings.parquet")



In [ ]:
df.describe()

In [3]:
df.head(3)

,id,document,embedding,metadata
0,14069121_0,a card was opened under my name by a fraudster...,"[-0.04277738183736801, 0.025624370202422142, -...","{'chunk_index': 0, 'company': 'CITIBANK, N.A.'..."
1,14061897_0,i made the mistake of using my wellsfargo debi...,"[-0.05458317697048187, 0.10340359061956406, 0....","{'chunk_index': 0, 'company': 'WELLS FARGO & C..."
2,14061897_1,and got a letter stating my dispute was reject...,"[-0.03491289168596268, 0.059216588735580444, 0...","{'chunk_index': 1, 'company': 'WELLS FARGO & C..."


In [5]:
df['document'].head()

0    a card was opened under my name by a fraudster...
1    i made the mistake of using my wellsfargo debi...
2    and got a letter stating my dispute was reject...
3    dear cfpb, i have a secured credit card with c...
4    y confirmation whatsoever to report to the pol...
Name: document, dtype: str

In [4]:
#Convert the embedding column to a continuous NumPy matrix
import numpy as np
print("Extracting embedding matrix...")
embeddings_matrix = np.vstack(df['embedding'].values).astype('float32')

Extracting embedding matrix...


In [6]:
#Instantiate the FAISS index
dimension = embeddings_matrix.shape[1]
print(f"Embedding dimension detected: {dimension}")

Embedding dimension detected: 384


In [7]:
import faiss
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_matrix)
print(f"Successfully indexed {index.ntotal} vector chunks into FAISS.")

Successfully indexed 1375327 vector chunks into FAISS.


In [8]:
#Save the serialized FAISS Index Binary
faiss.write_index(index, "../vector_store/index.faiss")

In [9]:
#Extract and align text + metadata attributes
print("Building explicit metadata mapping records...")
metadata_map = []

for idx, row in df.iterrows():
    # Gather everything needed to track back the source in the RAG UI
    meta_record = {
        "id": row['id'],                               # The baseline unique chunk/complaint ID
        "document": row['document'],                   # The actual text snippet of the narrative
        "chunk_index": row['metadata'].get('chunk_index'),
        "company": row['metadata'].get('company'),
        "product_category": row['metadata'].get('product_category', 'Credit Card') # Fallback if nested
    }
    metadata_map.append(meta_record)

Building explicit metadata mapping records...


In [10]:
import pickle
with open("../vector_store/index.pkl", "wb") as f:
    pickle.dump(metadata_map, f)

In [11]:
seed_database(parquet_path= '../data/raw/complaint_embeddings.parquet')

Loading embeddings from ../data/raw/complaint_embeddings.parquet...
Preparing batches for ChromaDB...


InternalError: Error in compaction: Error constructing hnsw segment reader: Error creating hnsw segment reader: Error loading hnsw index

In [12]:
eval_questions = [
    "What are the most common reasons customers report credit card fraud?",
    "Are customers experiencing unexpected fees when transferring money?",
    "What complaints exist regarding delayed processing times for Personal Loans?"
]

pipeline = RAGPipeline()
results = []

NotFoundError: Collection [cfpb_complaints] does not exist